
# Faster R-CNN — Two-stage Anchor-based Object Detection

This notebook implements a pretrained Faster R-CNN with a ResNet-50 FPN backbone on the VisDrone2019-VID dataset.

The experimental protocol uses sequence-level 3-fold cross-validation, frame sampling with step 6, batch size 2, and 6 training epochs.

The first run trains fold 0 using SGD. Training checkpoints, configuration details, and evaluation results are stored in Google Drive.


In [7]:

!pip -q install kaggle pycocotools scikit-learn

import os
import json
import time
import random
import shutil
import zipfile
import subprocess
import numpy as np
import pandas as pd
import torch
import torchvision

from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

SEED = 42
FRAME_STEP = 6
N_FOLDS = 3
BATCH_SIZE = 2
EPOCHS = 6
FOLD = 0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

ROOT = Path("/content/drive/MyDrive/UCU_AI_HW2")
ROOT.mkdir(parents=True, exist_ok=True)

SAVE_DIR = ROOT / "faster_rcnn" / f"fold_{FOLD}"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Device:", DEVICE)
print("Google Drive:", ROOT)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PyTorch: 2.11.0+cu130
Torchvision: 0.26.0+cu130
Device: cuda
Google Drive: /content/drive/MyDrive/UCU_AI_HW2
GPU: Tesla T4



## Dataset Storage

The competition dataset is downloaded using the Kaggle API and stored in Google Drive.

The archive is reused across Colab notebooks. Images are extracted to the local runtime for faster training, while model checkpoints are saved persistently to Google Drive.


In [8]:

from google.colab import userdata

COMPETITION = "multiple-object-video-detection-hw-2-ucu-ai-2026"
DRIVE_ZIP = ROOT / "visdrone_competition.zip"

# Use the existing Kaggle token without kaggle.json.
# If already configured in the environment, nothing changes.
if not os.environ.get("KAGGLE_API_TOKEN"):
    try:
        token = userdata.get("KAGGLE_API_TOKEN")
        if token:
            os.environ["KAGGLE_API_TOKEN"] = token
    except Exception:
        pass

if DRIVE_ZIP.exists() and zipfile.is_zipfile(DRIVE_ZIP):
    print("Existing valid dataset archive found.")
else:
    print("Downloading dataset to Google Drive...")

    subprocess.run([
        "kaggle",
        "competitions",
        "download",
        "-c", COMPETITION,
        "-p", str(ROOT),
    ], check=True)

    archives = [
        p for p in ROOT.glob("*.zip")
        if p.is_file()
    ]

    assert archives, "No downloaded ZIP found."

    source = max(archives, key=lambda p: p.stat().st_size)

    if source != DRIVE_ZIP:
        source.rename(DRIVE_ZIP)

assert zipfile.is_zipfile(DRIVE_ZIP), "Invalid archive."

print("Archive ready:", DRIVE_ZIP)
print("Size:", round(DRIVE_ZIP.stat().st_size / 1e9, 2), "GB")


Archive ready: /content/drive/MyDrive/UCU_AI_HW2/visdrone_competition.zip
Size: 10.61 GB


In [9]:

LOCAL_DIR = Path("/content/visdrone")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

matches = list(
    LOCAL_DIR.rglob("train/annotations_coco.json")
)

if not matches:
    print("Extracting dataset...")

    with zipfile.ZipFile(DRIVE_ZIP, "r") as archive:
        required = sum(
            f.file_size for f in archive.infolist()
        )

        free = shutil.disk_usage("/content").free

        print("Required:", round(required / 1e9, 2), "GB")
        print("Available:", round(free / 1e9, 2), "GB")

        assert free > required * 1.10, (
            "Insufficient local disk space."
        )

        archive.extractall(LOCAL_DIR)

    matches = list(
        LOCAL_DIR.rglob("train/annotations_coco.json")
    )

assert len(matches) > 0, (
    "COCO annotations not found. "
    "Inspect the ZIP structure."
)

DATA_ROOT = matches[0].parent.parent

assert (DATA_ROOT / "train/images").exists()
assert (DATA_ROOT / "val/images").exists()
assert (DATA_ROOT / "val/annotations_coco.json").exists()

with open(DATA_ROOT / "train/annotations_coco.json") as f:
    train_coco = json.load(f)

with open(DATA_ROOT / "val/annotations_coco.json") as f:
    holdout_coco = json.load(f)

print("DATA_ROOT:", DATA_ROOT)
print("Train frames:", len(train_coco["images"]))
print("Holdout frames:", len(holdout_coco["images"]))
print("Train annotations:", len(train_coco["annotations"]))

assert len(train_coco["images"]) == 24201
assert len(holdout_coco["images"]) == 2846

print("Dataset verification passed!")


Extracting dataset...
Required: 10.98 GB
Available: 110.19 GB
DATA_ROOT: /content/visdrone
Train frames: 24201
Holdout frames: 2846
Train annotations: 1616260
Dataset verification passed!



## Sequence-level Cross-Validation

Every sixth frame is selected independently within each training sequence.

GroupKFold with three folds prevents frames from the same video from appearing in both the training and validation portions of a fold.

The original validation set is reserved as an independent holdout set.


In [10]:

from sklearn.model_selection import GroupKFold

records = []

for img in train_coco["images"]:
    records.append({
        "image_id": img["id"],
        "file_name": img["file_name"],
        "clip": Path(img["file_name"]).parent.name,
    })

frames = pd.DataFrame(records)

frames = frames.sort_values(
    ["clip", "file_name"]
).reset_index(drop=True)

sampled = frames.loc[
    frames.groupby("clip").cumcount() % FRAME_STEP == 0
].copy().reset_index(drop=True)

gkf = GroupKFold(n_splits=N_FOLDS)
sampled["fold"] = -1

for fold, (_, val_idx) in enumerate(
    gkf.split(sampled, groups=sampled["clip"])
):
    sampled.loc[val_idx, "fold"] = fold

assert sampled["fold"].between(0, 2).all()
assert len(sampled) == 4062

train_ids = set(
    sampled.loc[sampled["fold"] != FOLD, "image_id"]
)
val_ids = set(
    sampled.loc[sampled["fold"] == FOLD, "image_id"]
)

train_clips = set(
    sampled.loc[sampled["fold"] != FOLD, "clip"]
)
val_clips = set(
    sampled.loc[sampled["fold"] == FOLD, "clip"]
)

assert train_ids.isdisjoint(val_ids)
assert train_clips.isdisjoint(val_clips)

print("Sampled frames:", len(sampled))
print("Fold distribution:")
print(sampled["fold"].value_counts().sort_index())

print("\nSelected fold:", FOLD)
print("Train frames:", len(train_ids))
print("Validation frames:", len(val_ids))
print("Overlapping clips:", len(train_clips & val_clips))

sampled.to_csv(
    ROOT / "cv_splits_step6.csv",
    index=False
)


Sampled frames: 4062
Fold distribution:
fold
0    1357
1    1356
2    1349
Name: count, dtype: int64

Selected fold: 0
Train frames: 2705
Validation frames: 1357
Overlapping clips: 0



## Data Loading and Annotation Conversion

COCO annotations are converted to the bounding box representation required by torchvision.

The model receives RGB images normalized to [0, 1], bounding boxes in xyxy format, and category labels from 1 to 8.

Ignored annotations are excluded from the training targets for the initial baseline. Their proper treatment requires a separate ignore-aware strategy.


In [11]:

from PIL import Image
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.functional import pil_to_tensor

annotations_by_image = defaultdict(list)

for ann in train_coco["annotations"]:
    annotations_by_image[ann["image_id"]].append(ann)

image_info = {
    img["id"]: img for img in train_coco["images"]
}

def resolve_image_path(info):
    file_name = Path(info["file_name"])

    candidates = [
        DATA_ROOT / "train" / file_name,
        DATA_ROOT / "train/images" /
            file_name.parent.name / file_name.name,
        DATA_ROOT / file_name,
    ]

    for path in candidates:
        if path.exists():
            return path

    raise FileNotFoundError(str(candidates))


class VisDroneDataset(Dataset):

    def __init__(self, image_ids):
        self.image_ids = sorted(image_ids)

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        info = image_info[image_id]

        image_path = resolve_image_path(info)
        image = Image.open(image_path).convert("RGB")

        width, height = image.size
        boxes, labels, areas = [], [], []

        for ann in annotations_by_image[image_id]:

            if ann.get("iscrowd", 0) == 1:
                continue

            cls = int(ann["category_id"])
            if cls not in range(1, 9):
                continue

            x, y, w, h = map(float, ann["bbox"])

            x1 = max(0, min(x, width))
            y1 = max(0, min(y, height))
            x2 = max(0, min(x + w, width))
            y2 = max(0, min(y + h, height))

            if x2 <= x1 or y2 <= y1:
                continue

            boxes.append([x1, y1, x2, y2])
            labels.append(cls)
            areas.append((x2 - x1) * (y2 - y1))

        boxes = torch.tensor(
            boxes, dtype=torch.float32
        ).reshape(-1, 4)

        labels = torch.tensor(
            labels, dtype=torch.int64
        )

        areas = torch.tensor(
            areas, dtype=torch.float32
        )

        image_tensor = pil_to_tensor(image).float() / 255.0

        target = {
            "boxes": boxes,
            "labels": labels,
            "area": areas,
            "image_id": torch.tensor(image_id),
            "iscrowd": torch.zeros(
                len(labels), dtype=torch.int64
            ),
        }

        return image_tensor, target


def collate_fn(batch):
    return tuple(zip(*batch))


train_dataset = VisDroneDataset(train_ids)
val_dataset = VisDroneDataset(val_ids)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn,
    pin_memory=True
)

images, targets = next(iter(train_loader))

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Image shape:", images[0].shape)
print("Boxes:", targets[0]["boxes"].shape)
print("Labels:", targets[0]["labels"][:10])

assert len(images) == BATCH_SIZE
assert all(torch.isfinite(t["boxes"]).all() for t in targets)

print("Dataset smoke test passed!")


Train batches: 1353
Validation batches: 679
Image shape: torch.Size([3, 1071, 1904])
Boxes: torch.Size([44, 4])
Labels: tensor([3, 3, 3, 8, 1, 4, 3, 3, 4, 1])
Dataset smoke test passed!



## Faster R-CNN Architecture and Training Configuration

A pretrained Faster R-CNN with a ResNet-50 FPN backbone is fine-tuned for eight target classes plus background.

The initial experiment uses SGD with momentum and cosine learning rate scheduling. RPN proposal counts and image resizing are fixed for the baseline.


In [12]:

from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn,
    FasterRCNN_ResNet50_FPN_Weights
)
from torchvision.models.detection.faster_rcnn import (
    FastRCNNPredictor
)

assert torch.cuda.is_available(), (
    "Enable a GPU runtime before model training."
)

DEVICE = torch.device("cuda")

model = fasterrcnn_resnet50_fpn(
    weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT,
    min_size=640,
    max_size=1024,
    box_detections_per_img=300,
    rpn_pre_nms_top_n_train=2000,
    rpn_post_nms_top_n_train=1000,
    rpn_pre_nms_top_n_test=1000,
    rpn_post_nms_top_n_test=500,
)

in_features = (
    model.roi_heads.box_predictor.cls_score.in_features
)

model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features,
    num_classes=9
)

model = model.to(DEVICE)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.002,
    momentum=0.9,
    weight_decay=0.0005
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

print("Model initialized.")
print("GPU:", torch.cuda.get_device_name(0))
print("Classes:", 9)


Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:00<00:00, 172MB/s]


Model initialized.
GPU: Tesla T4
Classes: 9


In [13]:

model.train()

images, targets = next(iter(train_loader))

images = [img.to(DEVICE) for img in images]

targets = [
    {k: v.to(DEVICE) for k, v in target.items()}
    for target in targets
]

with torch.autocast("cuda"):
    loss_dict = model(images, targets)
    total_loss = sum(loss_dict.values())

print("Loss components:")

for name, value in loss_dict.items():
    print(name, round(value.item(), 4))

print("Total loss:", round(total_loss.item(), 4))

assert torch.isfinite(total_loss)

del images, targets, loss_dict, total_loss
torch.cuda.empty_cache()

print("Smoke test passed!")


Loss components:
loss_classifier 1.976
loss_box_reg 0.5808
loss_objectness 0.1876
loss_rpn_box_reg 0.2302
Total loss: 2.9745
Smoke test passed!



## Baseline Training — Fold 0

The model is trained for six epochs using SGD, momentum, and cosine learning rate scheduling.

Training loss and learning rate are recorded after each epoch. Checkpoints are saved to Google Drive to allow training to resume after an interrupted Colab session.


In [14]:

from tqdm.auto import tqdm

CHECKPOINT_PATH = SAVE_DIR / "latest.pt"
HISTORY_PATH = SAVE_DIR / "history.csv"

scaler = torch.amp.GradScaler("cuda")

start_epoch = 0
history = []

# Resume from the last completed epoch
if CHECKPOINT_PATH.exists():
    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location=DEVICE,
        weights_only=False
    )

    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    scheduler.load_state_dict(checkpoint["scheduler"])

    if "scaler" in checkpoint:
        scaler.load_state_dict(checkpoint["scaler"])

    start_epoch = checkpoint["epoch"] + 1
    history = checkpoint["history"]

    print("Resuming from epoch:", start_epoch + 1)

for epoch in range(start_epoch, EPOCHS):

    model.train()
    start_time = time.monotonic()
    total_epoch_loss = 0.0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for images, targets in progress:

        images = [
            image.to(DEVICE) for image in images
        ]

        targets = [
            {
                key: value.to(DEVICE)
                for key, value in target.items()
            }
            for target in targets
        ]

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast("cuda"):
            losses = model(images, targets)
            total_loss = sum(losses.values())

        if not torch.isfinite(total_loss):
            raise RuntimeError("Non-finite training loss")

        scaler.scale(total_loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_epoch_loss += total_loss.item()

        progress.set_postfix(
            loss=f"{total_loss.item():.4f}"
        )

    scheduler.step()

    avg_loss = total_epoch_loss / len(train_loader)
    minutes = (time.monotonic() - start_time) / 60

    history.append({
        "epoch": epoch + 1,
        "train_loss": avg_loss,
        "learning_rate": optimizer.param_groups[0]["lr"],
        "minutes": minutes,
    })

    checkpoint = {
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "history": history,
        "fold": FOLD,
        "seed": SEED,
    }

    local_checkpoint = Path("/content/faster_rcnn_latest.pt")

    torch.save(checkpoint, local_checkpoint)

    temp_checkpoint = SAVE_DIR / "uploading.pt"
    shutil.copy2(local_checkpoint, temp_checkpoint)
    temp_checkpoint.replace(CHECKPOINT_PATH)

    pd.DataFrame(history).to_csv(
        HISTORY_PATH,
        index=False
    )

    print(
        f"Epoch {epoch + 1} complete | "
        f"loss={avg_loss:.4f} | "
        f"time={minutes:.1f} minutes"
    )

    print("Checkpoint saved to Google Drive.")

print("\nTRAINING FINISHED")
display(pd.DataFrame(history))


Epoch 1/6:   0%|          | 0/1353 [00:00<?, ?it/s]

Epoch 1 complete | loss=1.1086 | time=7.3 minutes
Checkpoint saved to Google Drive.


Epoch 2/6:   0%|          | 0/1353 [00:00<?, ?it/s]

Epoch 2 complete | loss=0.8799 | time=7.5 minutes
Checkpoint saved to Google Drive.


Epoch 3/6:   0%|          | 0/1353 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c40d23d37e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
Exception ignored in:     <function _MultiProcessingDataLoaderIter.__del__ at 0x7c40d23d37e0>if w.is_alive():

Traceback (most recent call last):
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
        self._shutdown_workers()assert self._parent_pid == os.getpid(), 'can only test a child process'
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers

    AssertionError: if w.is_alive():can only test a child process

  File "/usr/lib/

... (5 outputs left out)

Epoch 5 complete | loss=0.6477 | time=7.5 minutes
Checkpoint saved to Google Drive.


Epoch 6/6:   0%|          | 0/1353 [00:00<?, ?it/s]

Epoch 6 complete | loss=0.6182 | time=7.6 minutes
Checkpoint saved to Google Drive.

TRAINING FINISHED


,epoch,train_loss,learning_rate,minutes
0,1,1.108628,0.001866,7.302950
1,2,0.879864,0.001500,7.462307
2,3,0.775111,0.001000,7.499327
3,4,0.702958,0.000500,7.514760
4,5,0.647725,0.000134,7.538624
5,6,0.618163,0.000000,7.558209



## COCO-style Validation

Detection quality is evaluated using COCO metrics on the held-out fold.

Metrics include:
- mAP (IoU 0.50–0.95)
- AP50
- AP_small
- AP_medium
- AP_large
- Average inference time per frame

Ignored annotations are retained as crowd regions for COCO-style evaluation. The evaluation uses a maximum of 100 detections per image for standard COCO AP.


In [15]:

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from tqdm.auto import tqdm
import copy

# Load trained weights
assert CHECKPOINT_PATH.exists(), "Checkpoint not found."

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["model"])
model.eval()

# Build ground truth for the selected validation fold
gt_dataset = {
    "images": [
        img for img in train_coco["images"]
        if img["id"] in val_ids
    ],
    "annotations": [
        ann for ann in train_coco["annotations"]
        if ann["image_id"] in val_ids
    ],
    "categories": train_coco["categories"],
    "info": train_coco.get("info", {}),
    "licenses": train_coco.get("licenses", []),
}

coco_gt = COCO()
coco_gt.dataset = gt_dataset
coco_gt.createIndex()

predictions = []
inference_times = []

with torch.inference_mode():

    for images, targets in tqdm(
        val_loader, desc="COCO evaluation"
    ):

        images_gpu = [
            image.to(DEVICE) for image in images
        ]

        torch.cuda.synchronize()
        start = time.perf_counter()

        outputs = model(images_gpu)

        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start

        inference_times.append(
            elapsed / len(images)
        )

        for output, target in zip(outputs, targets):

            image_id = int(target["image_id"].item())

            boxes = output["boxes"].cpu().numpy()
            scores = output["scores"].cpu().numpy()
            labels = output["labels"].cpu().numpy()

            for box, score, label in zip(
                boxes, scores, labels
            ):

                if score < 0.001:
                    continue

                if not 1 <= int(label) <= 8:
                    continue

                x1, y1, x2, y2 = box

                predictions.append({
                    "image_id": image_id,
                    "category_id": int(label),
                    "bbox": [
                        float(x1),
                        float(y1),
                        float(x2 - x1),
                        float(y2 - y1),
                    ],
                    "score": float(score),
                })

assert predictions, "Model returned no predictions."

coco_dt = coco_gt.loadRes(predictions)

evaluator = COCOeval(
    coco_gt,
    coco_dt,
    iouType="bbox"
)

evaluator.params.imgIds = sorted(val_ids)
evaluator.params.catIds = list(range(1, 9))
evaluator.params.maxDets = [1, 10, 100]

evaluator.evaluate()
evaluator.accumulate()
evaluator.summarize()

results = {
    "model": "Faster R-CNN R50-FPN",
    "fold": FOLD,
    "epochs_completed": checkpoint["epoch"] + 1,
    "mAP50-95": float(evaluator.stats[0]),
    "AP50": float(evaluator.stats[1]),
    "AP75": float(evaluator.stats[2]),
    "AP_small": float(evaluator.stats[3]),
    "AP_medium": float(evaluator.stats[4]),
    "AP_large": float(evaluator.stats[5]),
    "inference_ms_per_frame": float(
        np.mean(inference_times) * 1000
    ),
}

results_df = pd.DataFrame([results])
results_df.to_csv(
    SAVE_DIR / "validation_metrics.csv",
    index=False
)

display(results_df)
print("Validation results saved.")


creating index...
index created!


COCO evaluation:   0%|          | 0/679 [00:00<?, ?it/s]

Loading and preparing results...
DONE (t=0.12s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=40.17s).
Accumulating evaluation results...
DONE (t=1.58s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.154
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.304
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.140
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.048
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.125
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.342
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.085
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.222
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.264
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

,model,fold,epochs_completed,mAP50-95,AP50,AP75,AP_small,AP_medium,AP_large,inference_ms_per_frame
0,Faster R-CNN R50-FPN,0,6,0.154145,0.303931,0.139557,0.048235,0.124787,0.34181,142.035246


Validation results saved.



## Faster R-CNN — Completing 3-Fold Cross-Validation

Fold 0 has already been trained and evaluated. The remaining folds (1 and 2) are trained independently using the same architecture, hyperparameters, sampling strategy, and training budget.

Each fold starts from the original COCO-pretrained weights. Model checkpoints and validation metrics are saved separately to Google Drive.


In [16]:

import gc
import time
import random
import shutil
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn,
    FasterRCNN_ResNet50_FPN_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

assert torch.cuda.is_available(), "Enable GPU in Colab!"
assert len(sampled) == 4062
assert {"image_id", "clip", "fold"}.issubset(sampled.columns)

DEVICE = torch.device("cuda")
CV_FOLDS = [1, 2]

# Reuse the same hyperparameters as fold 0
CV_EPOCHS = 6
CV_BATCH_SIZE = 2
CV_LR = 0.002
CV_SEED = 42

CV_ROOT = ROOT / "faster_rcnn"
CV_ROOT.mkdir(parents=True, exist_ok=True)

# Release GPU references from the previous run
for name in [
    "model", "optimizer", "scheduler", "scaler",
    "loss_dict", "total_loss", "losses",
    "images", "targets", "outputs", "images_gpu"
]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

print("GPU:", torch.cuda.get_device_name(0))
print("Folds to train:", CV_FOLDS)
print("Previous fold 0:", CV_ROOT / "fold_0")

assert (
    CV_ROOT / "fold_0" / "validation_metrics.csv"
).exists(), "Fold 0 validation metrics not found!"

print("Preparation successful!")


GPU: Tesla T4
Folds to train: [1, 2]
Previous fold 0: /content/drive/MyDrive/UCU_AI_HW2/faster_rcnn/fold_0
Preparation successful!


In [17]:

def set_fold_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def build_faster_rcnn():
    model = fasterrcnn_resnet50_fpn(
        weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT,
        min_size=640,
        max_size=1024,
        box_detections_per_img=300,
        rpn_pre_nms_top_n_train=2000,
        rpn_post_nms_top_n_train=1000,
        rpn_pre_nms_top_n_test=1000,
        rpn_post_nms_top_n_test=500,
    )

    in_features = (
        model.roi_heads.box_predictor
        .cls_score.in_features
    )

    model.roi_heads.box_predictor = FastRCNNPredictor(
        in_features,
        num_classes=9,
    )

    return model.to(DEVICE)


def get_fold_loaders(fold):
    fold_train = set(
        sampled.loc[
            sampled["fold"] != fold, "image_id"
        ]
    )

    fold_val = set(
        sampled.loc[
            sampled["fold"] == fold, "image_id"
        ]
    )

    train_clips = set(
        sampled.loc[sampled["fold"] != fold, "clip"]
    )
    val_clips = set(
        sampled.loc[sampled["fold"] == fold, "clip"]
    )

    assert fold_train.isdisjoint(fold_val)
    assert train_clips.isdisjoint(val_clips)

    train_ds = VisDroneDataset(fold_train)
    val_ds = VisDroneDataset(fold_val)

    train_dl = DataLoader(
        train_ds,
        batch_size=CV_BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        collate_fn=collate_fn,
        pin_memory=True,
    )

    val_dl = DataLoader(
        val_ds,
        batch_size=CV_BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        collate_fn=collate_fn,
        pin_memory=True,
    )

    return train_dl, val_dl, fold_val


def evaluate_fold(model, val_loader, val_ids, fold):
    model.eval()

    gt_dataset = {
        "images": [
            img for img in train_coco["images"]
            if img["id"] in val_ids
        ],
        "annotations": [
            ann for ann in train_coco["annotations"]
            if ann["image_id"] in val_ids
        ],
        "categories": train_coco["categories"],
        "info": train_coco.get("info", {}),
        "licenses": train_coco.get("licenses", []),
    }

    coco_gt = COCO()
    coco_gt.dataset = gt_dataset
    coco_gt.createIndex()

    predictions = []
    inference_times = []

    with torch.inference_mode():
        for images, targets in tqdm(
            val_loader,
            desc=f"Fold {fold} evaluation"
        ):
            images_gpu = [
                image.to(DEVICE) for image in images
            ]

            torch.cuda.synchronize()
            start = time.perf_counter()

            outputs = model(images_gpu)

            torch.cuda.synchronize()
            elapsed = time.perf_counter() - start

            inference_times.append(
                elapsed / len(images)
            )

            for output, target in zip(outputs, targets):
                image_id = int(target["image_id"].item())

                boxes = output["boxes"].cpu().numpy()
                scores = output["scores"].cpu().numpy()
                labels = output["labels"].cpu().numpy()

                for box, score, label in zip(
                    boxes, scores, labels
                ):
                    if score < 0.001:
                        continue

                    if not 1 <= int(label) <= 8:
                        continue

                    x1, y1, x2, y2 = box

                    predictions.append({
                        "image_id": image_id,
                        "category_id": int(label),
                        "bbox": [
                            float(x1),
                            float(y1),
                            float(x2 - x1),
                            float(y2 - y1),
                        ],
                        "score": float(score),
                    })

    assert predictions, "No predictions generated!"

    coco_dt = coco_gt.loadRes(predictions)

    evaluator = COCOeval(
        coco_gt, coco_dt, iouType="bbox"
    )

    evaluator.params.imgIds = sorted(val_ids)
    evaluator.params.catIds = list(range(1, 9))
    evaluator.params.maxDets = [1, 10, 100]

    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()

    result = {
        "model": "Faster R-CNN R50-FPN",
        "fold": fold,
        "epochs_completed": CV_EPOCHS,
        "mAP50-95": float(evaluator.stats[0]),
        "AP50": float(evaluator.stats[1]),
        "AP75": float(evaluator.stats[2]),
        "AP_small": float(evaluator.stats[3]),
        "AP_medium": float(evaluator.stats[4]),
        "AP_large": float(evaluator.stats[5]),
        "inference_ms_per_frame": float(
            np.mean(inference_times) * 1000
        ),
    }

    return result


In [18]:

from tqdm.auto import tqdm

for fold in CV_FOLDS:

    print("\n" + "=" * 60)
    print(f"STARTING FASTER R-CNN — FOLD {fold}")
    print("=" * 60)

    set_fold_seed(CV_SEED)

    fold_dir = CV_ROOT / f"fold_{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    checkpoint_path = fold_dir / "latest.pt"
    history_path = fold_dir / "history.csv"
    metrics_path = fold_dir / "validation_metrics.csv"

    train_dl, val_dl, fold_val_ids = get_fold_loaders(fold)

    print("Train images:", len(train_dl.dataset))
    print("Validation images:", len(val_dl.dataset))

    model = build_faster_rcnn()

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=CV_LR,
        momentum=0.9,
        weight_decay=0.0005,
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=CV_EPOCHS,
    )

    scaler = torch.amp.GradScaler("cuda")

    start_epoch = 0
    history = []

    if checkpoint_path.exists():
        checkpoint = torch.load(
            checkpoint_path,
            map_location=DEVICE,
            weights_only=False,
        )

        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scheduler.load_state_dict(checkpoint["scheduler"])

        if "scaler" in checkpoint:
            scaler.load_state_dict(checkpoint["scaler"])

        start_epoch = checkpoint["epoch"] + 1
        history = checkpoint["history"]

        print("Resuming from epoch:", start_epoch + 1)

    for epoch in range(start_epoch, CV_EPOCHS):

        model.train()
        epoch_start = time.monotonic()
        running_loss = 0.0

        progress = tqdm(
            train_dl,
            desc=f"Fold {fold} | Epoch {epoch+1}/{CV_EPOCHS}",
        )

        for images, targets in progress:

            images = [
                img.to(DEVICE) for img in images
            ]

            targets = [
                {k: v.to(DEVICE) for k, v in t.items()}
                for t in targets
            ]

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast("cuda"):
                loss_dict = model(images, targets)
                loss = sum(loss_dict.values())

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Non-finite loss at fold {fold}, "
                    f"epoch {epoch + 1}"
                )

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()

            progress.set_postfix(
                loss=f"{loss.item():.4f}"
            )

        scheduler.step()

        avg_loss = running_loss / len(train_dl)
        minutes = (
            time.monotonic() - epoch_start
        ) / 60

        history.append({
            "epoch": epoch + 1,
            "train_loss": avg_loss,
            "learning_rate": optimizer.param_groups[0]["lr"],
            "minutes": minutes,
        })

        checkpoint = {
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "history": history,
            "seed": CV_SEED,
            "fold": fold,
        }

        local_path = Path(
            f"/content/faster_rcnn_fold_{fold}.pt"
        )

        torch.save(checkpoint, local_path)

        temp_path = fold_dir / "uploading.pt"
        shutil.copy2(local_path, temp_path)
        temp_path.replace(checkpoint_path)

        pd.DataFrame(history).to_csv(
            history_path,
            index=False,
        )

        print(
            f"Fold {fold}, epoch {epoch+1} finished | "
            f"loss={avg_loss:.4f} | "
            f"time={minutes:.1f} min"
        )

    print(f"\nEvaluating fold {fold}...")

    result = evaluate_fold(
        model,
        val_dl,
        fold_val_ids,
        fold,
    )

    pd.DataFrame([result]).to_csv(
        metrics_path,
        index=False,
    )

    print(f"\nFOLD {fold} COMPLETED")
    print("mAP50-95:", result["mAP50-95"])
    print("AP50:", result["AP50"])
    print("AP_small:", result["AP_small"])

    del model, optimizer, scheduler, scaler
    del train_dl, val_dl

    gc.collect()
    torch.cuda.empty_cache()

print("\nALL REQUESTED FOLDS COMPLETED!")



STARTING FASTER R-CNN — FOLD 1
Train images: 2706
Validation images: 1356


Fold 1 | Epoch 1/6:   0%|          | 0/1353 [00:00<?, ?it/s]

Fold 1, epoch 1 finished | loss=1.1579 | time=7.3 min


Fold 1 | Epoch 2/6:   0%|          | 0/1353 [00:00<?, ?it/s]

Fold 1, epoch 2 finished | loss=0.9277 | time=7.4 min


Fold 1 | Epoch 3/6:   0%|          | 0/1353 [00:00<?, ?it/s]

... (23 outputs left out)

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c40d23d37e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c40d23d37e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

Fold 2, epoch 6 finished | loss=0.6529 | time=7.7 min

Evaluating fold 2...
creating index...
index created!


Fold 2 evaluation:   0%|          | 0/675 [00:00<?, ?it/s]

Loading and preparing results...
DONE (t=0.13s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=38.52s).
Accumulating evaluation results...
DONE (t=2.28s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.144
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.285
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.132
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.093
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.137
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.281
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.094
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.202
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.239
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

In [19]:

all_results = []

for fold in range(3):

    metrics_path = (
        CV_ROOT / f"fold_{fold}" / "validation_metrics.csv"
    )

    if not metrics_path.exists():
        print(f"Fold {fold}: results not yet available")
        continue

    df = pd.read_csv(metrics_path)
    all_results.append(df)

assert len(all_results) == 3, (
    "Not all three folds have evaluation results!"
)

cv_results = pd.concat(
    all_results,
    ignore_index=True
)

metric_columns = [
    "mAP50-95",
    "AP50",
    "AP75",
    "AP_small",
    "AP_medium",
    "AP_large",
    "inference_ms_per_frame",
]

summary = pd.DataFrame({
    "mean": cv_results[metric_columns].mean(),
    "std": cv_results[metric_columns].std(),
})

display(cv_results)
display(summary.round(4))

cv_results.to_csv(
    CV_ROOT / "cv_3fold_results.csv",
    index=False,
)

summary.to_csv(
    CV_ROOT / "cv_3fold_summary.csv",
)

print("3-fold cross-validation results saved!")


,model,fold,epochs_completed,mAP50-95,AP50,AP75,AP_small,AP_medium,AP_large,inference_ms_per_frame
0,Faster R-CNN R50-FPN,0,6,0.154145,0.303931,0.139557,0.048235,0.124787,0.341810,142.035246
1,Faster R-CNN R50-FPN,1,6,0.138959,0.274943,0.123234,0.032046,0.125930,0.345527,120.361299
2,Faster R-CNN R50-FPN,2,6,0.143584,0.285075,0.131638,0.092722,0.137193,0.280843,142.220415


,mean,std
mAP50-95,0.1456,0.0078
AP50,0.2880,0.0147
AP75,0.1315,0.0082
AP_small,0.0577,0.0314
AP_medium,0.1293,0.0069
AP_large,0.3227,0.0363
inference_ms_per_frame,134.8723,12.5673


3-fold cross-validation results saved!
